# 03 — Time-based Blind SQL Injection

> **`vuln_class`:** `SQL_INJ_TIME` · **Риск:** 8/10 · **CWE-89** · **CAPEC-7 (Blind SQLi)**

Атакующий **не видит ответ** запроса напрямую (например, API возвращает только 200/500). Зато он **управляет временем ответа** — `pg_sleep(N)` или тяжёлые вычисления внутри `CASE WHEN ...`.

По задержке атакующий перебирает биты пароля.


## 🧒 Аналогия для ребёнка

Представь телефонный автомат, который **молчит** — но если ты
наберёшь правильный номер, он **гудит дольше**, а если неправильный —
гудит коротко. Ты не видишь ничего, кроме длины гудка. Но
если ты будешь подбирать цифры одну за другой и смотреть,
где гудок стал длиннее — ты узнаешь весь номер.

В SQL: вместо «длинного гудка» — `pg_sleep(5)`. Вместо «правильной
цифры» — правильный бит пароля. Это медленно, но **работает на
тихом API, который даже не возвращает данные**.


## 1. Setup — API на одну функцию `update_last_seen(uid)`

Это типичный endpoint: «обнови время последнего захода пользователя».
Возвращает только OK/ERROR — никаких данных.


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Создаёт мок-БД с двумя таблицами: users и last_seen.
def setup_users_lastseen_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, login TEXT, password TEXT)")
    cur.execute("CREATE TABLE last_seen (user_id INTEGER PRIMARY KEY, ts TEXT)")
    cur.executemany(
        "INSERT INTO users (login, password) VALUES (?, ?)",
        [("admin", "SuperSecret123"), ("ivanov", "qwerty"), ("petrova", "12345")],
    )
    conn.commit()
    return conn


##
# @brief Имитируем pg_sleep(N) — SQLite не имеет такой функции,
#        поэтому мы её эмулируем через time.sleep().
# @details
#   В реальном PostgreSQL `pg_sleep(N)` — это встроенная функция,
#   и её можно вызвать прямо из запроса:
#       SELECT pg_sleep(5) FROM ...
#   Здесь мы парсим SQL, ловим `pg_sleep(N)` и делаем time.sleep до
#   выполнения. Концептуально это даёт тот же эффект — задержку,
#   управляемую SQL-payload-ом.
def execute_with_pg_sleep_emulation(conn, sql):
    m = re.search(r"pg_sleep\((\d+(?:\.\d+)?)\)", sql, re.IGNORECASE)
    delay = float(m.group(1)) if m else 0
    sql_clean = re.sub(r"pg_sleep\([^)]*\)", "1", sql, flags=re.IGNORECASE)
    if delay:
        time.sleep(min(delay, 2))  # обрезаем до 2 сек, чтобы ноутбук не висел
    return conn.execute(sql_clean).fetchall()


conn = setup_users_lastseen_db()
print("Setup готов — есть таблица users и last_seen.")


## 2. Уязвимая функция

Эндпоинт не возвращает данные — только OK. Атакующий не может
вытянуть данные напрямую, но может… **дождаться задержки**.


In [ ]:
##
# @brief УЯЗВИМАЯ функция: обновляет last_seen, но конкатенирует uid.
# @param uid  ID пользователя (от клиента — текстом).
# @return     "ok" или "error", больше ничего.
# @warning    Time-based blind injection.
def update_last_seen_BAD(conn, uid: str):
    sql = f"UPDATE last_seen SET ts = datetime('now') WHERE user_id = {uid}"
    print(f"  SQL: {sql}")
    t0 = time.time()
    try:
        execute_with_pg_sleep_emulation(conn, sql)
        dt = time.time() - t0
        print(f"  Ответ: ok (время: {dt:.2f} сек)")
        return "ok", dt
    except Exception as e:
        print(f"  Ответ: error — {e}")
        return "error", 0


section("Нормальный вызов")
update_last_seen_BAD(conn, "1")


## 3. Атака — boolean-exfiltration по времени

Payload: `1 OR (CASE WHEN <условие> THEN pg_sleep(2) ELSE 0 END)`.

Если условие истинно — задержка 2 сек. Иначе — мгновенно.
Перебирая условие по битам, атакующий вытаскивает пароль.


In [ ]:
section("АТАКА: «угадываем первый символ пароля admin»")

# Атакующий не знает пароль. Перебирает по букве.
candidates = ["S", "P", "1", "q"]  # первая буква пароля
for c in candidates:
    payload = f"1 OR (CASE WHEN (SELECT password FROM users WHERE login='admin') LIKE '{c}%' THEN pg_sleep(2) ELSE 0 END)"
    print(f"\n  Проверяем символ '{c}'...")
    _, dt = update_last_seen_BAD(conn, payload)
    if dt > 1.5:
        print(f"  💀 Задержка > 1.5 сек → первый символ пароля admin = '{c}'")


## 4. Аудитор Phase 1 — `R006-pg-sleep`

В проде через pglast обходим AST и ищем `FuncCall` с именами
`pg_sleep`, `pg_sleep_for`, `pg_sleep_until`. Здесь — regex.


In [ ]:
##
# @brief Phase 1 правило R006 — детект pg_sleep в SQL.
def audit_R006_pg_sleep(sql_text):
    findings = []
    if re.search(r"\bpg_sleep(_for|_until)?\s*\(", sql_text, re.IGNORECASE):
        # Если внутри CASE WHEN ... THEN pg_sleep — это классический blind
        is_case = bool(re.search(r"CASE\s+WHEN.*?pg_sleep", sql_text,
                                 re.IGNORECASE | re.DOTALL))
        findings.append({
            "rule_id":       "R006-pg-sleep",
            "vuln_class":    "SQL_INJ_TIME",
            "severity":      "high", "risk_score": 9 if is_case else 8,
            "message":       "pg_sleep() в SQL — индикатор blind-injection"
                            + (" (внутри CASE — почти 100% blind exfil)" if is_case else ""),
            "evidence_refs": ["CWE-89", "CAPEC-7"],
        })
    return findings


section("Аудитор анализирует payload атакующего")
malicious = "UPDATE last_seen SET ts=datetime('now') WHERE user_id = 1 OR (CASE WHEN ... THEN pg_sleep(2) ELSE 0 END)"
for f in audit_R006_pg_sleep(malicious):
    print_finding(f)


## 5. Безопасная версия

Параметризация + ограничение `statement_timeout` на роли БД.


In [ ]:
##
# @brief Безопасная функция: параметризация + ожидаемый int.
# @note  В проде дополнительно: ALTER ROLE app SET statement_timeout = '5s'.
def update_last_seen_GOOD(conn, uid: int):
    sql = "UPDATE last_seen SET ts = datetime('now') WHERE user_id = ?"
    conn.execute(sql, (int(uid),))  # ← int() кастует, не-число выбросит ValueError
    return "ok"


section("АТАКА на безопасную версию")
payload = "1 OR pg_sleep(2)"
try:
    update_last_seen_GOOD(conn, payload)
except ValueError as e:
    print(f"  ✅ Безопасная версия отвергла ввод: {e}")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/03-sql-injection-time-blind/README.md](../../problems/vulnerabilities/03-sql-injection-time-blind/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/03-sql-injection-time-blind/solutions.md](../../problems/vulnerabilities/03-sql-injection-time-blind/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
